## Refactoring Checklist

### Issues Found:
- [ x] Function `process_multiple_products` does too much:
      it loops through products, extracts product data, encodes images, calls the API,
      handles errors, waits between calls, creates folders, and saves JSON files.

- [ x] Function `generate_product_listing` depends on the global `client`.
      This makes the function harder to test and reuse.

- [x] API settings are hardcoded in `generate_product_listing`:
      model `"gpt-4o"`, `max_tokens=1000`, and `response_format`.

- [x] Product price is hardcoded as `29.99` in multiple API calls.

- [x] Delay is hardcoded as `time.sleep(10)` inside `process_multiple_products`.

- [x] Output paths are hardcoded:
      `"results"`, `"listings.json"`, and `"errors.json"`.

- [x ] Error handling exists, but it is very general:
      `except Exception as e` catches everything without separating API errors,
      JSON parsing errors, missing product fields, or file-writing errors.

- [ x] Validation is missing for the API response.
      The code assumes the response always contains `"title"`, `"description"`,
      `"features"`, and `"keywords"`.

- [ x] Dataset fields are accessed directly:
      `sample["productDisplayName"]`, `sample["masterCategory"]`,
      `sample["baseColour"]`, `sample["season"]`.
      If one field is missing, the code breaks.

- [ x] Code for extracting product information is repeated in the single-product test
      and in `process_multiple_products`.

- [x ] The notebook runs API calls directly in cells.
      This makes it harder to reuse the code as a clean Python script.

- [ ] There is no `main()` function to control the program flow.

- [ ] File writing does not specify encoding.

### Priority:
1. Split `process_multiple_products` into smaller helper functions.
2. Move hardcoded values into constants or function parameters.
3. Add validation for API response and product fields.
4. Improve error handling with clearer error messages.
5. Remove global dependencies like `client` from functions.
6. Add a `main()` function for a cleaner script structure.




In [89]:
from datasets import load_dataset
from pathlib import Path
from io import BytesIO
from openai import OpenAI
from dotenv import load_dotenv
import base64, json, time, pandas as pd


In [91]:
MODEL = "gpt-4o"
MAX_TOKENS = 1000
DEFAULT_PRICE = 29.99
DELAY_SECONDS = 10

DATASET_NAME = "ashraq/fashion-product-images-small"
DATASET_SPLIT = "train[:100]"

OUTPUT_DIR = Path("results")
LISTINGS_FILE = "listings.json"
ERRORS_FILE = "errors.json"


In [92]:
load_dotenv()
client = OpenAI()

dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
print(f"✓ {len(dataset)} products loaded")
print(f"  Columns: {dataset.column_names}")

✓ 100 products loaded
  Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']


In [93]:
def encode_image_to_base64(image):
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")

# Checkpoint
encoded = encode_image_to_base64(dataset[0]["image"])
print(f"✓ Encoded | Length: {len(encoded)} | Preview: {encoded[:50]}...")

✓ Encoded | Length: 2400 | Preview: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQ...


In [94]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    additional = f"- Additional Info: {additional_info}" if additional_info else ""
    return f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{additional}

Format your response as JSON:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2"
}}"""

# Checkpoint
test = create_product_listing_prompt("Headphones", 79.99, "Electronics", "Noise cancelling")
assert "79.99" in test
assert "Electronics" in test
print("✓ Prompt template OK")

✓ Prompt template OK


In [95]:
def parse_api_response(response):
    """Parse JSON content from an OpenAI API response."""
    try:
        content = response.choices[0].message.content
        return json.loads(content)

    except json.JSONDecodeError as e:
        error_msg = (
            f"ERROR in parse_api_response(): JSONDecodeError\n"
            f"  Location: API response content, line {e.lineno}, column {e.colno}\n"
            f"  Message: {e.msg}\n"
            f"  Suggestion: Check whether the API returned valid JSON."
        )
        print(error_msg)
        raise

    except (AttributeError, IndexError) as e:
        error_msg = (
            f"ERROR in parse_api_response(): {type(e).__name__}\n"
            f"  Location: response.choices[0].message.content\n"
            f"  Message: {e}\n"
            f"  Suggestion: Check the structure of the OpenAI API response."
        )
        print(error_msg)
        raise


In [96]:
def generate_product_listing(client, image_base64, product_name, price, category, additional_info=None):
    prompt = create_product_listing_prompt(product_name, price, category, additional_info)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                {"type": "text", "text": prompt}
            ]
        }],
        max_tokens=MAX_TOKENS,
        response_format={"type": "json_object"}
    )
    return parse_api_response(response)


In [98]:
def extract_product_info(sample, default_price=DEFAULT_PRICE):
    """Extract product information from one dataset sample."""
    return {
        "id": sample.get("id", "unknown_id"),
        "product_name": sample.get("productDisplayName", "Unknown product"),
        "price": default_price,
        "category": sample.get("masterCategory", "Unknown category"),
        "additional_info": f"{sample.get('baseColour', 'Unknown color')}, {sample.get('season', 'Unknown season')}"
    }


In [100]:
def save_json(data, file_path):
    """Save data as a JSON file with error handling."""
    file_path = Path(file_path)

    try:
        file_path.parent.mkdir(exist_ok=True)

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    except PermissionError as e:
        error_msg = (
            f"ERROR in save_json(): PermissionError\n"
            f"  Location: File '{file_path}'\n"
            f"  Message: {e}\n"
            f"  Suggestion: Check that you have permission to write to this folder."
        )
        print(error_msg)
        raise

    except TypeError as e:
        error_msg = (
            f"ERROR in save_json(): TypeError\n"
            f"  Location: Data for file '{file_path}'\n"
            f"  Message: {e}\n"
            f"  Suggestion: Check that the data can be converted to JSON."
        )
        print(error_msg)
        raise

    except OSError as e:
        error_msg = (
            f"ERROR in save_json(): {type(e).__name__}\n"
            f"  Location: File '{file_path}'\n"
            f"  Message: {e}\n"
            f"  Suggestion: Check the file path and disk access."
        )
        print(error_msg)
        raise


In [101]:
def validate_listing(listing):
    """Validate that the API listing has the required fields."""
    required_fields = ["title", "description", "features", "keywords"]

    for field in required_fields:
        if field not in listing:
            raise ValueError(f"Missing field in listing: {field}")

    if not isinstance(listing["features"], list):
        raise ValueError("Field 'features' must be a list")

    return True


In [102]:
def process_single_product(sample, client):
    """Process one product sample and return the generated listing result."""
    product_info = extract_product_info(sample)

    listing = generate_product_listing(
        client=client,
        image_base64=encode_image_to_base64(sample["image"]),
        product_name=product_info["product_name"],
        price=product_info["price"],
        category=product_info["category"],
        additional_info=product_info["additional_info"]
    )

    validate_listing(listing)

    return {
        "id": product_info["id"],
        "product_name": product_info["product_name"],
        "listing": listing
    }

In [103]:
def format_error(product_info, error, function_name):
    """Format error details with useful context."""
    return {
        "id": product_info["id"],
        "product_name": product_info["product_name"],
        "function": function_name,
        "error_type": type(error).__name__,
        "error_message": str(error),
        "suggestion": "Check the product data, API response, or API connection."
    }

In [104]:
def process_multiple_products(dataset, client, num_products=3):
    results, errors = [], []

    for i in range(num_products):
        sample = dataset[i]
        product_info = extract_product_info(sample)
        product_name = product_info["product_name"]

        print(f"Processing {i+1}/{num_products}: {product_name}...")

        try:
            result = process_single_product(sample, client)
            results.append(result)

            print(f"  ✓ {result['listing']['title']}")

        except Exception as e:
            error_info = format_error(product_info, e, "process_multiple_products")
            errors.append(error_info)
            print(f"  ⚠ {e}")

        time.sleep(DELAY_SECONDS)

    save_json(results, OUTPUT_DIR / LISTINGS_FILE)
    save_json(errors, OUTPUT_DIR / ERRORS_FILE)

    return results, errors


results, errors = process_multiple_products(dataset, client, num_products=3)
print(f"Processed: {len(results)}")
print(f"Errors: {len(errors)}")

Processing 1/3: Turtle Check Men Navy Blue Shirt...
  ✓ Turtle Check Men Navy Blue Shirt - Stylish Fall Apparel
Processing 2/3: Peter England Men Party Blue Jeans...
  ✓ Peter England Men's Party Blue Jeans - Stylish Comfort for Any Occasion
Processing 3/3: Titan Women Silver Watch...
  ✓ Elegant Titan Women Silver Watch - Perfect Winter Accessory
Processed: 3
Errors: 0


In [6]:
assert len(results) > 0, "Keine Ergebnisse — siehe errors"
assert Path("results/listings.json").exists()

print(f"✓ Processed: {len(results)} | ⚠ Errors: {len(errors)}")
print("\n" + "="*50)
print(json.dumps(results[0]["listing"], indent=2))

✓ Processed: 3 | ⚠ Errors: 0

{
  "title": "Turtle Check Men's Navy Blue Shirt - Perfect for Fall",
  "description": "Elevate your fall wardrobe with the Turtle Check Men's Navy Blue Shirt. Crafted for comfort and style, this shirt features a timeless check pattern, ideal for both casual outings and smart-casual settings. Made with high-quality materials, it ensures durability and a perfect fit that keeps you looking sharp all season long.",
  "features": [
    "Classic check pattern",
    "Comfortable fit and durable fabric",
    "Versatile style for various occasions"
  ],
  "keywords": "men's navy blue shirt, check pattern shirt, Turtle check shirt, fall fashion, casual wear"
}


## Step 6: Test Refactored Code

The refactored code was tested with valid and invalid inputs to confirm that:
- valid data is processed correctly
- missing fields are detected
- invalid API JSON responses show clear errors
- invalid JSON saving data shows clear errors


In [105]:
# Test: Extract product information
test_product_info = extract_product_info(dataset[0])
print(test_product_info)

{'id': 15970, 'product_name': 'Turtle Check Men Navy Blue Shirt', 'price': 29.99, 'category': 'Apparel', 'additional_info': 'Navy Blue, Fall'}


In [106]:
valid_listing = {
    "title": "Test Product",
    "description": "A short product description.",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "test, product"
}

print(validate_listing(valid_listing))

True


In [107]:
invalid_listing = {
    "description": "Missing title.",
    "features": ["Feature 1"],
    "keywords": "test"
}

try:
    validate_listing(invalid_listing)
except ValueError as e:
    print(f"Caught expected error: {e}")


Caught expected error: Missing field in listing: title


In [108]:
# Test: format_error helper
test_product_info = extract_product_info(dataset[0])

try:
    raise ValueError("Missing field in listing: title")
except ValueError as e:
    test_error = format_error(test_product_info, e, "test_function")
    print(test_error)

{'id': 15970, 'product_name': 'Turtle Check Men Navy Blue Shirt', 'function': 'test_function', 'error_type': 'ValueError', 'error_message': 'Missing field in listing: title', 'suggestion': 'Check the product data, API response, or API connection.'}


In [109]:
#Test: Invalid API JSON Response
class FakeBadMessage:
    content = "not valid json"

class FakeBadChoice:
    message = FakeBadMessage()

class FakeBadResponse:
    choices = [FakeBadChoice()]

try:
    parse_api_response(FakeBadResponse())
except json.JSONDecodeError:
    print("Caught expected JSONDecodeError")


ERROR in parse_api_response(): JSONDecodeError
  Location: API response content, line 1, column 1
  Message: Expecting value
  Suggestion: Check whether the API returned valid JSON.
Caught expected JSONDecodeError


In [110]:
#Test: Invalid JSON Save Data
try:
    save_json({"bad_data": {1, 2, 3}}, OUTPUT_DIR / "bad_test.json")
except TypeError:
    print("Caught expected TypeError")


ERROR in save_json(): TypeError
  Location: Data for file 'results/bad_test.json'
  Message: Object of type set is not JSON serializable
  Suggestion: Check that the data can be converted to JSON.
Caught expected TypeError


In [111]:
# Test: Single real API call
sample = dataset[0]

test_listing = generate_product_listing(
    client=client,
    image_base64=encode_image_to_base64(sample["image"]),
    product_name=sample["productDisplayName"],
    price=DEFAULT_PRICE,
    category=sample["masterCategory"],
    additional_info=f"{sample['baseColour']}, {sample['season']}"
)

validate_listing(test_listing)

print("✓ API Call OK")
print(f"  Title: {test_listing['title']}")

✓ API Call OK
  Title: Turtle Check Men’s Navy Blue Shirt - Stylish Fall Collection


In [112]:
#Test: Full processing with one product
results, errors = process_multiple_products(dataset, client, num_products=1)

print(f"Processed: {len(results)}")
print(f"Errors: {len(errors)}")
print(errors)

Processing 1/1: Turtle Check Men Navy Blue Shirt...
  ✓ Turtle Check Men's Navy Blue Shirt - Perfect for Fall
Processed: 1
Errors: 0
[]


In [113]:
#Test: Check output files
print((OUTPUT_DIR / LISTINGS_FILE).exists())
print((OUTPUT_DIR / ERRORS_FILE).exists())


True
True


### Test Summary

- Inline checkpoint tests inside the function sections were removed to keep the notebook structure clean.
- All tests were moved to the Step 6 testing section.
- Product information extraction was tested successfully with `extract_product_info()`.
- Valid listing validation passed with `validate_listing()`.
- Invalid listing validation showed a clear missing-field error.
- Error formatting was tested successfully with `format_error()`.
- Invalid API response parsing showed the JSON error location and a helpful suggestion.
- Invalid JSON saving showed the file path, error type, and suggestion.
- The full processing workflow was tested with one product using `process_multiple_products()`.
- Output files were created successfully.
